# 面试问题：怎样从零实现 BiLSTM-CRF，并解释 forward、NLL 与 Viterbi？

## 可直接复述的回答主线

1. BiLSTM 用前向和反向 LSTM 同时编码左右上下文，再为每个 token 输出各标签 emission 分数。
2. CRF 在 emission 之外学习相邻标签转移，路径得分由 START、逐 token emission、转移和 END 四部分组成。
3. forward algorithm 用 log-sum-exp 动态规划计算全部合法路径的 log-partition，NLL 等于 log-partition 减 gold 路径分数。
4. Viterbi 把 log-sum-exp 换成 max，并保存 backpointer，最终回溯出全局最高分路径。
5. BIO 约束应同时进入分母和解码，不能只在后处理时修补非法的 I 标签。
6. 评估要展示逐句 token、gold、baseline、Viterbi、emission 与路径分数，而不是只看 loss 或 shape。
7. 生产还需子词对齐、多实体类型、partial label、长序列截断、CRF 加速、置信度校准和实体级漂移监控。

下面用同一批可读输入依次验证朴素基线、手写核心机制、中间过程、失败修正与生产边界。

## 1. 真实案例与输入预览

案例是 8 条脱敏中文电商客服短句，任务是抽取商品名称，标签为 O、B-PROD、I-PROD。句子覆盖单词商品、品牌加型号、两个商品并列和不同长度；数据是无需下载的教学样本，只验证计算图能学习这组结构，不代表线上 NER 泛化。

In [1]:
import math  # 计算梯度范数并汇总训练指标。
import warnings  # 过滤本地 PyTorch 环境的无关兼容警告。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 保持 notebook 输出聚焦教学结果。
import torch  # 使用基础张量与自动微分手写 LSTM 和 CRF。
torch.manual_seed(211)  # 固定参数初始化与训练轨迹。
torch.set_num_threads(1)  # 固定 CPU 单线程并缩短冷启动训练时间。
tag_names = ["O", "B-PROD", "I-PROD"]  # 定义简化 BIO 商品标签空间。
raw_samples = [{"id": "q01", "tokens": ["用户", "想", "购买", "华为", "Mate", "60"], "tags": ["O", "O", "O", "B-PROD", "I-PROD", "I-PROD"]}, {"id": "q02", "tokens": ["客户", "咨询", "苹果", "iPhone", "15", "价格"], "tags": ["O", "O", "B-PROD", "I-PROD", "I-PROD", "O"]}, {"id": "q03", "tokens": ["门店", "有", "小米", "14", "现货"], "tags": ["O", "O", "B-PROD", "I-PROD", "O"]}, {"id": "q04", "tokens": ["请", "比较", "荣耀", "Magic", "6", "和", "华为", "Pura", "70"], "tags": ["O", "O", "B-PROD", "I-PROD", "I-PROD", "O", "B-PROD", "I-PROD", "I-PROD"]}, {"id": "q05", "tokens": ["退货", "商品", "是", "联想", "小新", "Pro"], "tags": ["O", "O", "O", "B-PROD", "I-PROD", "I-PROD"]}, {"id": "q06", "tokens": ["查一下", "大疆", "Mini", "4", "库存"], "tags": ["O", "B-PROD", "I-PROD", "I-PROD", "O"]}, {"id": "q07", "tokens": ["我要", "预订", "索尼", "A7M4"], "tags": ["O", "O", "B-PROD", "I-PROD"]}, {"id": "q08", "tokens": ["取消", "戴森", "吹风机", "订单"], "tags": ["O", "B-PROD", "I-PROD", "O"]}]  # 定义八条有业务语义的商品抽取样本。
vocabulary = {"<PAD>": 0, "<UNK>": 1}  # 初始化 padding 与未知词编号。
for sample in raw_samples:  # 扫描全部客服 token 构建教学词表。
    for token in sample["tokens"]:  # 逐 token 分配稳定编号。
        if token not in vocabulary:  # 仅为首次出现的 token 建立条目。
            vocabulary[token] = len(vocabulary)  # 使用当前词表长度作为新编号。
tag_to_id = {name: index for index, name in enumerate(tag_names)}  # 建立标签到整数的映射。
maximum_length = max(len(sample["tokens"]) for sample in raw_samples)  # 读取批次最大句长供右侧 padding。
token_ids = torch.zeros(len(raw_samples), maximum_length, dtype=torch.long)  # 初始化批次 token 编号。
tag_ids = torch.zeros(len(raw_samples), maximum_length, dtype=torch.long)  # 初始化批次标签编号。
mask = torch.zeros(len(raw_samples), maximum_length, dtype=torch.bool)  # 标记每句真实 token 位置。
for row, sample in enumerate(raw_samples):  # 逐句写入变长批次张量。
    length = len(sample["tokens"])  # 读取当前句真实长度。
    token_ids[row, :length] = torch.tensor([vocabulary[token] for token in sample["tokens"]])  # 写入当前句 token 编号。
    tag_ids[row, :length] = torch.tensor([tag_to_id[tag] for tag in sample["tags"]])  # 写入当前句 BIO 编号。
    mask[row, :length] = True  # 将当前句真实位置设为有效。
print("教学实验输入：8条商品NER客服句，batch shapes=", tuple(token_ids.shape), tuple(tag_ids.shape), tuple(mask.shape))  # 展示数据规模和变长张量形状。
for sample in raw_samples:  # 逐句展示读者可理解的 token 与标签。
    print(f"{sample['id']} 句子={' | '.join(sample['tokens'])}  gold={' | '.join(sample['tags'])}")  # 输出当前原始记录和目标序列。

教学实验输入：8条商品NER客服句，batch shapes= (8, 9) (8, 9) (8, 9)
q01 句子=用户 | 想 | 购买 | 华为 | Mate | 60  gold=O | O | O | B-PROD | I-PROD | I-PROD
q02 句子=客户 | 咨询 | 苹果 | iPhone | 15 | 价格  gold=O | O | B-PROD | I-PROD | I-PROD | O
q03 句子=门店 | 有 | 小米 | 14 | 现货  gold=O | O | B-PROD | I-PROD | O
q04 句子=请 | 比较 | 荣耀 | Magic | 6 | 和 | 华为 | Pura | 70  gold=O | O | B-PROD | I-PROD | I-PROD | O | B-PROD | I-PROD | I-PROD
q05 句子=退货 | 商品 | 是 | 联想 | 小新 | Pro  gold=O | O | O | B-PROD | I-PROD | I-PROD
q06 句子=查一下 | 大疆 | Mini | 4 | 库存  gold=O | B-PROD | I-PROD | I-PROD | O
q07 句子=我要 | 预订 | 索尼 | A7M4  gold=O | O | B-PROD | I-PROD
q08 句子=取消 | 戴森 | 吹风机 | 订单  gold=O | B-PROD | I-PROD | O


## 2. Baseline / 基线：所有 token 都预测为 O

实体 token 通常少于非实体，全部预测 O 会得到看似不低的 token accuracy，却抽不出任何商品实体。这个基线与 BiLSTM-CRF 使用完全相同的八条句子。

In [2]:
baseline_predictions = torch.zeros_like(tag_ids)  # 为每个有效 token 预测标签 O。
baseline_correct = ((baseline_predictions == tag_ids) & mask).sum()  # 统计有效位置预测正确数。
baseline_token_accuracy = float((baseline_correct / mask.sum()).item())  # 计算全部 O 的 token accuracy。
baseline_exact_sentences = sum(int(torch.equal(baseline_predictions[row, :int(mask[row].sum())], tag_ids[row, :int(mask[row].sum())])) for row in range(len(raw_samples)))  # 统计整句标签完全正确数。
print("Baseline：全部预测 O")  # 标记下表为类别不平衡基线。
for row, sample in enumerate(raw_samples):  # 逐句显示基线预测而非只给汇总数字。
    length = int(mask[row].sum())  # 读取当前句有效长度。
    predicted_tags = [tag_names[index] for index in baseline_predictions[row, :length].tolist()]  # 把基线编号还原为标签名。
    correct_tokens = int((baseline_predictions[row, :length] == tag_ids[row, :length]).sum())  # 统计当前句正确 token 数。
    print(f"{sample['id']} baseline={' | '.join(predicted_tags)}  token_correct={correct_tokens}/{length}")  # 输出当前句基线序列和准确数。
print(f"Baseline token accuracy={baseline_token_accuracy:.4f}，exact sentences={baseline_exact_sentences}/{len(raw_samples)}")  # 展示类别不平衡带来的误导性准确率。

Baseline：全部预测 O
q01 baseline=O | O | O | O | O | O  token_correct=3/6
q02 baseline=O | O | O | O | O | O  token_correct=3/6
q03 baseline=O | O | O | O | O  token_correct=3/5
q04 baseline=O | O | O | O | O | O | O | O | O  token_correct=3/9
q05 baseline=O | O | O | O | O | O  token_correct=3/6
q06 baseline=O | O | O | O | O  token_correct=2/5
q07 baseline=O | O | O | O  token_correct=2/4
q08 baseline=O | O | O | O  token_correct=2/4
Baseline token accuracy=0.4667，exact sentences=0/8


## 3. 底层实现：手写 LSTM 扫描、CRF Forward Algorithm 与 Viterbi

`transition[next, previous]` 明确固定方向。Forward Algorithm 对所有合法路径做 `logsumexp`；Viterbi 对相同状态图取 `max` 并保存 backpointer。START 不能进入 I，O 也不能直接转入 I。

In [3]:
class ScratchLSTMCell(torch.nn.Module):  # 定义不调用 nn.LSTM 的单步四门 LSTM。
    def __init__(self, input_size, hidden_size):  # 初始化拼接输入到四门的线性参数。
        super().__init__()  # 注册 PyTorch 模块参数。
        self.hidden_size = hidden_size  # 保存隐藏维度供状态初始化。
        self.gates = torch.nn.Linear(input_size + hidden_size, 4 * hidden_size)  # 一次计算输入门、遗忘门、候选和输出门。
    def forward(self, inputs, state):  # 对一个时间步更新隐藏与记忆状态。
        hidden, memory = state  # 解包上一时刻两种状态。
        input_gate, forget_gate, candidate, output_gate = self.gates(torch.cat([inputs, hidden], dim=1)).chunk(4, dim=1)  # 按 ifgo 顺序切分门向量。
        input_gate = torch.sigmoid(input_gate)  # 把输入门映射到零到一。
        forget_gate = torch.sigmoid(forget_gate)  # 把遗忘门映射到零到一。
        candidate = torch.tanh(candidate)  # 生成有符号候选记忆。
        output_gate = torch.sigmoid(output_gate)  # 把输出门映射到零到一。
        next_memory = forget_gate * memory + input_gate * candidate  # 合并保留记忆与新候选。
        next_hidden = output_gate * torch.tanh(next_memory)  # 生成当前隐藏状态。
        return next_hidden, next_memory  # 返回供下一时间步使用的状态。
class ScratchBiLSTM(torch.nn.Module):  # 定义显式前向与反向变长扫描器。
    def __init__(self, input_size, hidden_size):  # 初始化两个方向独立 LSTM cell。
        super().__init__()  # 注册两个方向的参数。
        self.hidden_size = hidden_size  # 保存单向隐藏维度。
        self.forward_cell = ScratchLSTMCell(input_size, hidden_size)  # 创建从左到右的 cell。
        self.backward_cell = ScratchLSTMCell(input_size, hidden_size)  # 创建从右到左的 cell。
    def scan(self, inputs, valid_mask, cell, reverse=False):  # 在一个方向上逐时间步扫描。
        batch_size, time_steps, _ = inputs.shape  # 读取批量与序列长度。
        hidden = inputs.new_zeros(batch_size, self.hidden_size)  # 初始化隐藏状态为零。
        memory = inputs.new_zeros(batch_size, self.hidden_size)  # 初始化记忆状态为零。
        outputs = [None] * time_steps  # 预留每个原始位置的输出槽。
        positions = range(time_steps - 1, -1, -1) if reverse else range(time_steps)  # 根据方向生成时间索引。
        for position in positions:  # 显式遍历序列位置。
            active = valid_mask[:, position].unsqueeze(1)  # 读取当前批次哪些句子仍有效。
            candidate_hidden, candidate_memory = cell(inputs[:, position], (hidden, memory))  # 计算当前时间步候选状态。
            hidden = torch.where(active, candidate_hidden, hidden)  # 只更新真实 token 对应的隐藏状态。
            memory = torch.where(active, candidate_memory, memory)  # 只更新真实 token 对应的记忆状态。
            outputs[position] = torch.where(active, hidden, torch.zeros_like(hidden))  # padding 位置输出严格为零。
        return torch.stack(outputs, dim=1)  # 恢复批次乘时间乘隐藏布局。
    def forward(self, inputs, valid_mask):  # 同时执行两个方向并拼接上下文。
        forward_hidden = self.scan(inputs, valid_mask, self.forward_cell, reverse=False)  # 从左到右编码前文。
        backward_hidden = self.scan(inputs, valid_mask, self.backward_cell, reverse=True)  # 从右到左编码后文。
        return torch.cat([forward_hidden, backward_hidden], dim=2)  # 在特征维拼接双向表示。
class LinearChainCRF(torch.nn.Module):  # 定义手写线性链 CRF 的训练与解码。
    def __init__(self, tag_count, allowed_start, allowed_transitions):  # 初始化可训练路径参数和 BIO 约束。
        super().__init__()  # 注册 CRF 参数。
        self.tag_count = tag_count  # 保存标签数供动态规划使用。
        self.transitions = torch.nn.Parameter(torch.empty(tag_count, tag_count))  # 定义 next 乘 previous 转移矩阵。
        self.start = torch.nn.Parameter(torch.empty(tag_count))  # 定义 START 到首标签的分数。
        self.end = torch.nn.Parameter(torch.empty(tag_count))  # 定义末标签到 END 的分数。
        torch.nn.init.uniform_(self.transitions, -0.1, 0.1)  # 小范围初始化标签转移。
        torch.nn.init.uniform_(self.start, -0.1, 0.1)  # 小范围初始化起点分数。
        torch.nn.init.uniform_(self.end, -0.1, 0.1)  # 小范围初始化终点分数。
        self.register_buffer("allowed_start", allowed_start)  # 保存合法首标签布尔向量。
        self.register_buffer("allowed_transitions", allowed_transitions)  # 保存合法相邻标签布尔矩阵。
    def constrained_parameters(self):  # 把非法路径替换为足够小的有限分数。
        transitions = self.transitions.masked_fill(~self.allowed_transitions, -1.0e4)  # 约束相邻标签转移。
        start = self.start.masked_fill(~self.allowed_start, -1.0e4)  # 约束句首不能为 I。
        return transitions, start, self.end  # 返回动态规划实际使用的三组参数。
    def log_partition(self, emissions, valid_mask, return_trace=False):  # 用 forward algorithm 求全部路径 log-partition。
        transitions, start, end = self.constrained_parameters()  # 取得带 BIO 门禁的参数。
        alpha = start + emissions[:, 0]  # 初始化首位置每个标签的路径总分。
        trace = [alpha]  # 保存 alpha 轨迹供教学观察。
        for position in range(1, emissions.shape[1]):  # 从第二个 token 开始动态规划。
            candidate_scores = alpha[:, None, :] + transitions[None, :, :]  # 构造每个 previous 到 next 的候选分数。
            next_alpha = torch.logsumexp(candidate_scores, dim=2) + emissions[:, position]  # 对 previous 维做稳定 log-sum-exp。
            alpha = torch.where(valid_mask[:, position, None], next_alpha, alpha)  # 句子结束后冻结其状态。
            trace.append(alpha)  # 保存当前时间步全部标签的 alpha。
        partition = torch.logsumexp(alpha + end, dim=1)  # 汇总最后标签到 END 的全部路径。
        return (partition, torch.stack(trace, dim=1)) if return_trace else partition  # 按需返回中间 alpha 张量。
    def gold_score(self, emissions, tags, valid_mask):  # 计算给定 gold 序列的完整路径分数。
        transitions, start, end = self.constrained_parameters()  # 取得带约束路径参数。
        batch_indices = torch.arange(emissions.shape[0])  # 构造批次索引用于 gather。
        score = start[tags[:, 0]] + emissions[batch_indices, 0, tags[:, 0]]  # 加入 START 与首位置 emission。
        for position in range(1, emissions.shape[1]):  # 累加后续真实 token 的 emission 与转移。
            step_score = transitions[tags[:, position], tags[:, position - 1]] + emissions[batch_indices, position, tags[:, position]]  # 计算当前位置 gold 增量。
            score = score + step_score * valid_mask[:, position]  # padding 位置不进入路径分数。
        last_positions = valid_mask.sum(dim=1) - 1  # 找到每句最后一个真实 token 位置。
        last_tags = tags[batch_indices, last_positions]  # 读取每句末标签。
        return score + end[last_tags]  # 加入末标签到 END 的分数。
    def neg_log_likelihood(self, emissions, tags, valid_mask):  # 计算批次平均条件负对数似然。
        return (self.log_partition(emissions, valid_mask) - self.gold_score(emissions, tags, valid_mask)).mean()  # 用 logZ 减 gold 路径分数。
    def viterbi(self, emissions, valid_mask):  # 用 max-product 动态规划解码最高分合法路径。
        transitions, start, end = self.constrained_parameters()  # 取得带 BIO 门禁的参数。
        score = start + emissions[:, 0]  # 初始化首位置标签分数。
        backpointers = []  # 保存每一步 next 标签对应的最佳 previous。
        for position in range(1, emissions.shape[1]):  # 从第二个 token 递推最佳路径。
            candidates = score[:, None, :] + transitions[None, :, :]  # 构造全部 previous 到 next 候选。
            best_score, best_previous = candidates.max(dim=2)  # 对 previous 维取最大并保存来源。
            best_score = best_score + emissions[:, position]  # 加入当前位置 emission。
            score = torch.where(valid_mask[:, position, None], best_score, score)  # 对已结束句子冻结分数。
            backpointers.append(best_previous)  # 保存当前时间步回溯指针。
        final_score, final_tag = (score + end).max(dim=1)  # 选择每句最佳末标签与路径分数。
        paths = []  # 保存逐句可变长标签路径。
        for row, length_tensor in enumerate(valid_mask.sum(dim=1)):  # 逐句按真实长度回溯。
            length = int(length_tensor.item())  # 把当前长度转为 Python 整数。
            tag = int(final_tag[row].item())  # 读取当前最佳末标签。
            path = [tag]  # 用末标签初始化反向路径。
            for position in range(length - 1, 0, -1):  # 从末位置回溯到第二个位置。
                tag = int(backpointers[position - 1][row, tag].item())  # 读取当前标签的最佳前驱。
                path.append(tag)  # 把前驱标签加入反向路径。
            paths.append(list(reversed(path)))  # 翻转得到正向 Viterbi 路径。
        return paths, final_score  # 返回逐句标签列表和路径分数。
class BiLSTMCRF(torch.nn.Module):  # 组合 token embedding、手写 BiLSTM、emission 和 CRF。
    def __init__(self, vocabulary_size, tag_count):  # 初始化完整序列标注模型。
        super().__init__()  # 注册四个核心子模块。
        self.embedding = torch.nn.Embedding(vocabulary_size, 12, padding_idx=0)  # 把 token 编号映射到十二维向量。
        self.encoder = ScratchBiLSTM(12, 16)  # 创建双向十六维上下文编码器。
        self.emission = torch.nn.Linear(32, tag_count)  # 把双向表示映射到三个标签分数。
        allowed_start = torch.tensor([True, True, False])  # 禁止 I-PROD 出现在句首。
        allowed_transitions = torch.tensor([[True, True, True], [True, True, True], [False, True, True]])  # 禁止 previous=O 时 next=I-PROD。
        self.crf = LinearChainCRF(tag_count, allowed_start, allowed_transitions)  # 创建带 BIO 约束的 CRF。
    def forward(self, tokens, valid_mask, tags=None, return_debug=False):  # 统一训练 NLL 与推理解码入口。
        embeddings = self.embedding(tokens)  # 生成批次 token embedding。
        contextual = self.encoder(embeddings, valid_mask)  # 执行手写双向变长扫描。
        emissions = self.emission(contextual)  # 生成每个 token 的三个 emission 分数。
        if tags is not None:  # 检查当前调用是否为监督训练。
            loss = self.crf.neg_log_likelihood(emissions, tags, valid_mask)  # 计算 CRF 条件 NLL。
            return (loss, {"embeddings": embeddings, "contextual": contextual, "emissions": emissions}) if return_debug else loss  # 按需返回训练中间量。
        paths, scores = self.crf.viterbi(emissions, valid_mask)  # 对推理请求执行全局 Viterbi。
        return (paths, scores, {"embeddings": embeddings, "contextual": contextual, "emissions": emissions}) if return_debug else paths  # 按需返回解码证据。
model = BiLSTMCRF(len(vocabulary), len(tag_names))  # 创建待训练的完整 BiLSTM-CRF。
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)  # 创建同时更新 LSTM、emission 和 CRF 的优化器。
history = []  # 保存真实 backward 的损失与梯度轨迹。
for step in range(300):  # 对八条教学句执行全批次过拟合训练。
    optimizer.zero_grad(set_to_none=True)  # 清除上一步全部参数梯度。
    loss, training_debug = model(token_ids, mask, tag_ids, return_debug=True)  # 前向计算 CRF 负对数似然。
    loss.backward()  # 对 embedding、双向 LSTM、emission 和 CRF 反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters() if parameter.grad is not None))  # 汇总全部非空梯度二范数。
    optimizer.step()  # 使用 Adam 应用真实参数更新。
    if step % 75 == 0 or step == 299:  # 每七十五步保存可读训练状态。
        history.append({"step": step, "nll": loss.item(), "gradient_norm": gradient_norm})  # 记录损失与梯度证据。
model.eval()  # 切换到确定性推理模式。
with torch.no_grad():  # 在无梯度环境执行最终 Viterbi 与 forward trace。
    decoded_paths, decoded_scores, inference_debug = model(token_ids, mask, return_debug=True)  # 解码全部八条客服句。
    partitions, alpha_trace = model.crf.log_partition(inference_debug["emissions"], mask, return_trace=True)  # 取得每步 alpha 与 logZ。
print("BiLSTM-CRF训练轨迹=", history)  # 展示 NLL 下降和非零梯度。
print("q01 emission前三个token=", torch.round(inference_debug["emissions"][0, :3] * 1000) / 1000)  # 展示标签局部分数矩阵。
print("q01 forward alpha前三步=", torch.round(alpha_trace[0, :3] * 1000) / 1000)  # 展示 log-sum-exp 动态规划状态。
print("学习到的transition[next,prev]=", torch.round(model.crf.transitions.detach() * 1000) / 1000)  # 展示标签相邻依赖参数。

BiLSTM-CRF训练轨迹= [{'step': 0, 'nll': 4.94036865234375, 'gradient_norm': 2.404164580013268}, {'step': 75, 'nll': 0.00014734268188476562, 'gradient_norm': 0.00035354742613969326}, {'step': 150, 'nll': 0.00010585784912109375, 'gradient_norm': 0.0002564711516416488}, {'step': 225, 'nll': 7.748603820800781e-05, 'gradient_norm': 0.00018875480917702908}, {'step': 299, 'nll': 5.7697296142578125e-05, 'gradient_norm': 0.00014396989709569813}]
q01 emission前三个token= tensor([[  7.0360, -10.0370,   1.3390],
        [  8.7970,  -9.4410,  -1.6260],
        [  8.2670,  -8.0130,  -2.2190]])
q01 forward alpha前三步= tensor([[ 7.3720e+00, -1.0516e+01, -9.9987e+03],
        [ 1.5810e+01, -1.5880e+00, -1.1574e+01],
        [ 2.3718e+01,  8.2780e+00, -3.2380e+00]])
学习到的transition[next,prev]= tensor([[-0.3590, -0.4150,  0.6430],
        [ 0.4810, -0.4270, -0.5020],
        [ 0.0430,  0.5690,  0.1560]])


## 4. 逐句结果与结果解读

比较同一八条句子的全部 O 基线与 Viterbi 路径，并报告 token accuracy、整句完全正确数和每句路径分数。这里是受控过拟合实验，不是独立测试集成绩。

In [4]:
model_prediction_tensor = torch.zeros_like(tag_ids)  # 初始化便于统一计算的预测张量。
for row, path in enumerate(decoded_paths):  # 把每句可变长 Viterbi 列表写回批次张量。
    model_prediction_tensor[row, :len(path)] = torch.tensor(path)  # 仅覆盖真实 token 位置。
model_token_accuracy = float((((model_prediction_tensor == tag_ids) & mask).sum() / mask.sum()).item())  # 计算 Viterbi token accuracy。
model_exact_sentences = sum(int(torch.equal(model_prediction_tensor[row, :int(mask[row].sum())], tag_ids[row, :int(mask[row].sum())])) for row in range(len(raw_samples)))  # 统计完整标签序列正确句数。
print("id   tokens                              gold                               baseline                           BiLSTM-CRF                    score")  # 输出逐句对照表头。
for row, sample in enumerate(raw_samples):  # 逐句展示模型真正抽取的标签序列。
    length = int(mask[row].sum())  # 读取当前句真实长度。
    baseline_tags = [tag_names[index] for index in baseline_predictions[row, :length].tolist()]  # 还原基线标签名。
    predicted_tags = [tag_names[index] for index in decoded_paths[row]]  # 还原 Viterbi 标签名。
    print(f"{sample['id']:<4} {'/'.join(sample['tokens']):<35} {'/'.join(sample['tags']):<34} {'/'.join(baseline_tags):<34} {'/'.join(predicted_tags):<30} {decoded_scores[row].item():.3f}")  # 输出当前句 token、gold、两种预测和路径分数。
print(f"结果解读：全部O基线 accuracy={baseline_token_accuracy:.4f}、exact={baseline_exact_sentences}/8；BiLSTM-CRF accuracy={model_token_accuracy:.4f}、exact={model_exact_sentences}/8。")  # 解释局部类别不平衡与全局序列学习差异。

id   tokens                              gold                               baseline                           BiLSTM-CRF                    score
q01  用户/想/购买/华为/Mate/60                  O/O/O/B-PROD/I-PROD/I-PROD         O/O/O/O/O/O                        O/O/O/B-PROD/I-PROD/I-PROD     45.414
q02  客户/咨询/苹果/iPhone/15/价格               O/O/B-PROD/I-PROD/I-PROD/O         O/O/O/O/O/O                        O/O/B-PROD/I-PROD/I-PROD/O     41.987
q03  门店/有/小米/14/现货                       O/O/B-PROD/I-PROD/O                O/O/O/O/O                          O/O/B-PROD/I-PROD/O            28.637
q04  请/比较/荣耀/Magic/6/和/华为/Pura/70        O/O/B-PROD/I-PROD/I-PROD/O/B-PROD/I-PROD/I-PROD O/O/O/O/O/O/O/O/O                  O/O/B-PROD/I-PROD/I-PROD/O/B-PROD/I-PROD/I-PROD 49.571
q05  退货/商品/是/联想/小新/Pro                   O/O/O/B-PROD/I-PROD/I-PROD         O/O/O/O/O/O                        O/O/O/B-PROD/I-PROD/I-PROD     43.034
q06  查一下/大疆/Mini/4/库存                    O/B-PROD/I-PROD/I-PROD/O           O/

## 5. 失败案例与修正：逐 token argmax 产生非法句首 I-PROD

人为构造三个 token 的 emission，让首位置 I-PROD 分数最高。独立 argmax 会直接输出非法 I；同一 emission 经过 BIO 约束 Viterbi 后，首标签只能是 O 或 B-PROD。

In [5]:
adversarial_emissions = torch.tensor([[[0.0, 8.0, 10.0], [0.0, 1.0, 9.0], [7.0, 0.0, 1.0]]])  # 构造首位置偏爱非法 I 的局部分数。
adversarial_mask = torch.ones(1, 3, dtype=torch.bool)  # 标记三个位置均为真实 token。
naive_path = adversarial_emissions.argmax(dim=2)[0].tolist()  # 用逐 token argmax 复现无结构解码错误。
constrained_path, constrained_score = model.crf.viterbi(adversarial_emissions, adversarial_mask)  # 对完全相同 emission 执行 BIO Viterbi。
naive_names = [tag_names[index] for index in naive_path]  # 把错误路径编号还原为标签名。
constrained_names = [tag_names[index] for index in constrained_path[0]]  # 把修正路径编号还原为标签名。
print(f"错误行为：token argmax={naive_names}，首标签={naive_names[0]}，违反BIO句首约束。")  # 展示局部最优导致的非法路径。
print(f"修正行为：constrained Viterbi={constrained_names}，score={constrained_score[0].item():.3f}，首标签合法。")  # 展示状态图约束后的全局最优路径。

错误行为：token argmax=['I-PROD', 'I-PROD', 'O']，首标签=I-PROD，违反BIO句首约束。
修正行为：constrained Viterbi=['B-PROD', 'I-PROD', 'O']，score=25.450，首标签合法。


## 6. 生产边界

八句过拟合不能衡量未知商品、错别字或长文本。生产需用真实 train/dev/test 时间切分，处理 WordPiece 到字符 span 对齐、多实体 BIOES 约束、partial annotation、padding 批处理、混合精度稳定性、Viterbi 批量加速，并监控实体级 precision/recall、非法路径率、未知词率和业务词表漂移。

In [6]:
bilstm_crf_diagnostics = {"sentences": len(raw_samples), "tokens": int(mask.sum()), "vocabulary": len(vocabulary), "initial_nll": history[0]["nll"], "final_nll": history[-1]["nll"], "baseline_accuracy": baseline_token_accuracy, "model_accuracy": model_token_accuracy, "exact_sentences": model_exact_sentences, "naive_first_tag": naive_names[0], "viterbi_first_tag": constrained_names[0]}  # 汇总数据、训练、序列结果和失败分支。
print("生产监控快照：", bilstm_crf_diagnostics)  # 输出序列标注服务应持续观察的核心信号。

生产监控快照： {'sentences': 8, 'tokens': 45, 'vocabulary': 46, 'initial_nll': 4.94036865234375, 'final_nll': 5.7697296142578125e-05, 'baseline_accuracy': 0.46666666865348816, 'model_accuracy': 1.0, 'exact_sentences': 8, 'naive_first_tag': 'I-PROD', 'viterbi_first_tag': 'B-PROD'}


## 7. 最小回归测试

最后一格只保留数据规模、真实训练、解码收益、动态规划形状和 BIO 修正的关键不变量。

In [7]:
assert len(raw_samples) >= 6 and int(mask.sum()) >= 30  # 保证案例包含足够多可读句子与 token。
assert history[-1]["nll"] < history[0]["nll"] and all(row["gradient_norm"] > 0.0 for row in history)  # 保证完整 BiLSTM-CRF 实际 backward 学习。
assert model_token_accuracy > baseline_token_accuracy and model_token_accuracy >= 0.95  # 保证同数据序列模型明显优于全部 O 基线。
assert model_exact_sentences >= 7 and len(decoded_paths) == len(raw_samples)  # 保证绝大多数整句 Viterbi 标签完全正确。
assert alpha_trace.shape == (len(raw_samples), maximum_length, len(tag_names)) and torch.isfinite(partitions).all()  # 保证 forward algorithm 中间状态形状正确且有限。
assert naive_path[0] == tag_to_id["I-PROD"] and constrained_path[0][0] != tag_to_id["I-PROD"]  # 保证非法逐点路径可复现并被 BIO Viterbi 修正。